# 库的使用学习笔记

这份笔记专门介绍项目里主要外部库的职责、在代码中的位置，以及阅读代码时应该怎么看这些库的作用。

阅读目标：
- 理解主线为什么使用 `Gymnasium + MuJoCo + Stable-Baselines3`
- 理解 Warp 训练线为什么使用 `JAX + Brax + MuJoCo Playground + warp-lang`
- 把“库的名字”映射到“实际代码位置”


## 1. 主线使用的库

主线任务是单机版 `UR5` 到点训练线，重点是环境清晰、训练稳定、便于调试。

In [ ]:
import pandas as pd

main_lib_df = pd.DataFrame(
    [
        ['Gymnasium', '环境接口', '定义 reset/step/render 和 spaces', 'ur5_reach_env.py'],
        ['MuJoCo', '物理引擎', '加载 XML、推进动力学、碰撞检测、渲染', 'ur5_reach_env.py'],
        ['Stable-Baselines3', '算法框架', '提供 TD3/SAC/PPO、VecNormalize、EvalCallback', 'train_ur5_reach.py'],
        ['NumPy', '数组运算', '处理观测、动作、奖励和日志中的数值计算', 'ur5_reach_env.py / train_ur5_reach.py'],
    ],
    columns=['library', 'role', 'usage', 'files'],
)
main_lib_df


## 2. Warp 训练线使用的库

Warp 训练线的目标是纯 GPU 高吞吐训练，因此环境表示和训练器都和主线不同。

In [ ]:
warp_lib_df = pd.DataFrame(
    [
        ['JAX', '张量与函数式计算', '保存和更新环境状态、采样随机数、处理观测和奖励', 'warp_ur5_env.py'],
        ['Brax', '训练器', '提供 PPO/SAC 训练器和参数保存接口', 'train_ur5_reach_warp.py / test_ur5_reach_warp.py'],
        ['MuJoCo Playground', '环境适配层', '提供 MjxEnv 基类和 wrap_for_brax_training', 'warp_ur5_env.py / train_ur5_reach_warp.py'],
        ['warp-lang', 'GPU 后端运行时', '初始化 GPU 设备并提供 Warp 后端', 'warp_ur5_runtime.py'],
        ['mujoco-warp', 'MuJoCo Warp 适配', '让 MuJoCo 模型能运行在 Warp 实现上', 'warp_ur5_runtime.py / warp_ur5_env.py'],
        ['Flax / Orbax', '参数恢复', '恢复 checkpoint 参数树和标准化状态', 'test_ur5_reach_warp.py'],
    ],
    columns=['library', 'role', 'usage', 'files'],
)
warp_lib_df


## 3. 主线代码里怎么找到这些库

建议按这个顺序看：

1. 看 `ur5_reach_env.py`，理解 `Gymnasium` 和 `MuJoCo` 如何配合
2. 看 `train_ur5_reach.py`，理解 `Stable-Baselines3` 如何接入环境
3. 对照 `docs/IMPLEMENTATION_GUIDE.md`，把流程串起来


主线最关键的调用关系：
- `UR5ReachEnv(gym.Env)`：说明环境接口来自 Gymnasium
- `mujoco.MjModel.from_xml_path(...)`：说明物理模型来自 MuJoCo
- `TD3(...) / SAC(...) / PPO(...)`：说明训练算法来自 Stable-Baselines3
- `VecNormalize(...)`：说明训练时额外做了标准化包装


## 4. Warp 代码里怎么找到这些库

建议按这个顺序看：

1. 看 `warp_ur5_runtime.py`，理解 Warp 运行时怎么检查
2. 看 `warp_ur5_env.py`，理解 JAX / MJX / Warp 环境怎么组织
3. 看 `train_ur5_reach_warp.py`，理解 Brax 训练器怎么接进来
4. 最后看 `test_ur5_reach_warp.py`，理解参数恢复和推理怎么做


Warp 最关键的调用关系：
- `mjx_env.MjxEnv`：环境基类来自 MuJoCo Playground
- `mjx.put_model(..., impl='warp')`：MuJoCo 模型切到 Warp 后端
- `brax.training.agents.ppo.train` / `brax.training.agents.sac.train`：训练器来自 Brax
- `warp.init()` / `warp.set_device(...)`：运行时和设备管理来自 warp-lang


## 5. 为什么主线和 Warp 要用不同库

主线和 Warp 的差异不只是“算法不一样”，而是整套执行路径不同：

- 主线更适合稳定训练、逐步调试、窗口观察和教学阅读
- Warp 更适合大规模并行、GPU 吞吐和纯 JAX/Brax 训练流程

所以这两个分支共用同一个任务目标和同一个机器人模型，但训练基础设施不同。

## 6. 结合文档继续读

建议把这份笔记和下面几个文件一起看：

- `docs/LIBRARY_USAGE.md`
- `docs/IMPLEMENTATION_GUIDE.md`
- `docs/WARP_IMPLEMENTATION_GUIDE.md`
- `train_ur5_reach.py`
- `ur5_reach_env.py`
- `train_ur5_reach_warp.py`
- `warp_ur5_env.py`
